# Table 1: Wind-tunnel causal graph recovery

Use 20 ESNs for each input window, with $W\in\{1,2,4,6\}$ state blocks including the current state. Dependencies: NumPy, pandas and threadpoolctl.


In [ ]:
import hashlib
import json
import shutil
import types
import urllib.request
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import HTML, display
from threadpoolctl import threadpool_limits

SETTINGS = {'input_blocks': [1, 2, 4, 6], 'n_models': 20,
            'n_initial_histories': 30, 'n_evolution_steps': 5,
            'first_anchor': 110, 'mean_threshold': 0.01, 'batch_size': 16,
            'reservoir_seed': 8499, 'history_seed': 8599, 'blas_threads': 2}
MODEL_CONFIG = {'n_units': 128, 'input_scaling': 0.5, 'spectral_radius': 0.9,
                'leak': 0.3, 'ridge': 0.01}
TRAIN_IDS = list(range(1, 11))
VALIDATION_IDS = [11, 12, 13]
TEST_IDS = [14, 15, 16]
VARIABLES = ['hatch', 'load_in', 'load_out', 'pot_1', 'pot_2', 'current_in',
             'current_out', 'pressure_downwind', 'pressure_upwind', 'rpm_in',
             'rpm_out', 'mic', 'pressure_intake', 'pressure_ambient', 'signal_1', 'signal_2']
NODES = len(VARIABLES)
EDGE_MASK = ~np.eye(NODES, dtype=bool)
DATA_ROOT = Path('data') / 'CausalChambers'
RESULT_ROOT = Path('result') / 'Wind_tunnel'
SOURCE_URLS = {
    'archive': 'https://causalchamber.s3.eu-central-1.amazonaws.com/downloadables/wt_walks_v1.zip',
    'truth': 'https://raw.githubusercontent.com/juangamella/causal-chamber-package/9fb5d82e391bb91a89a64f2e1c9b6ae8e7aed6d6/causalchamber/ground_truth/adjacencies/adjacency_wt_standard.csv'}
SOURCE_SHA256 = {'wt_walks_v1.zip': '1ef92d39fff7645a9e6e8cd41311e708362606491590a1534c6a5135d207736e',
                 'adjacency_wt_standard.csv': '05088000010c094b9f6a29bcb62d67b154507395e54441ee7748eaf5ecd5024d'}


def token(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def implementation_id(*functions):
    def describe(code):
        return {'bytecode': code.co_code.hex(), 'names': code.co_names,
                'constants': [describe(value) if isinstance(value, types.CodeType) else repr(value)
                              for value in code.co_consts]}
    return token([describe(function.__code__) for function in functions])


def save_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2), encoding='utf-8')


def save_npz(path, **arrays):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix('.tmp.npz')
    np.savez_compressed(temporary, **arrays)
    temporary.replace(path)


def download(url, path):
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_suffix('.download')
        with urllib.request.urlopen(url, timeout=120) as source, temporary.open('wb') as target:
            shutil.copyfileobj(source, target)
        temporary.replace(path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != SOURCE_SHA256[path.name]:
        raise ValueError(f'Dataset checksum mismatch: {path.name}')


def model_seed(index):
    return int(np.random.SeedSequence([SETTINGS['reservoir_seed'], index]).generate_state(1)[0])


## 1. Data

Use official [Causal Chambers](https://github.com/juangamella/causal-chamber) `wt_walks_v1` records 1-10 for training, 11-13 for validation and 14-16 for testing. Normalize all records using the pooled training median and IQR (`midpoint` quantiles), without interpolation. Reset reservoir states at record boundaries and exclude the first 110 samples from training pairs.


In [ ]:
ARCHIVE = DATA_ROOT / 'wt_walks_v1.zip'
TRUTH_PATH = DATA_ROOT / 'adjacency_wt_standard.csv'
download(SOURCE_URLS['archive'], ARCHIVE)
download(SOURCE_URLS['truth'], TRUTH_PATH)


def read_record(record):
    with zipfile.ZipFile(ARCHIVE) as archive:
        with archive.open(f'wt_walks_v1/actuators_random_walk_{record}.csv') as stream:
            return pd.read_csv(stream)[VARIABLES].to_numpy(dtype=np.float64)


RAW_TRAIN = {record: read_record(record) for record in TRAIN_IDS}
POOLED_TRAIN = np.concatenate(list(RAW_TRAIN.values()))
Q1, MEDIAN, Q3 = np.percentile(POOLED_TRAIN, [25, 50, 75], axis=0, method='midpoint')
SCALE = np.where(Q3 > Q1, Q3 - Q1, 1.0)
TRAIN = {record: (values - MEDIAN) / SCALE for record, values in RAW_TRAIN.items()}
VALIDATION = {record: (read_record(record) - MEDIAN) / SCALE for record in VALIDATION_IDS}
TEST = {record: (read_record(record) - MEDIAN) / SCALE for record in TEST_IDS}
TRUTH = pd.read_csv(TRUTH_PATH, index_col=0).loc[VARIABLES, VARIABLES].to_numpy(dtype=int).T
TRAIN_HASHES = {str(record): hashlib.sha256(values.tobytes()).hexdigest()
                for record, values in TRAIN.items()}
del RAW_TRAIN, POOLED_TRAIN
save_npz(RESULT_ROOT / 'scaling.npz', median=MEDIAN, scale=SCALE, variables=np.asarray(VARIABLES))


## 2. ESN and DDI

The 128-unit leaky ESN uses $[x_t,x_{t-1},\ldots,x_{t-(W-1)}]$ to predict the next state $\widehat{x}_{t+1}$. The window contains $W$ state blocks, spanning $W-1$ sampling steps. Pad missing history with the first observation. Mask the source in current and historical inputs, replacing only the target output during intervention rollouts.

For each of 30 initial histories, divide the target's five-step mean squared response by the signed control-source mean, including the initial prediction. Exclude absolute denominators below 0.01, average strictly positive scores (zero if none), then average the 20 model-level DDI matrices. Rows index effects; columns index causes.


In [ ]:
class HistoryESN:
    def __init__(self, config, seed, history):
        self.config = dict(config)
        self.history = history
        self.driver_width = NODES * (history + 1)
        units = config['n_units']
        rng = np.random.RandomState(seed)
        self.W = rng.uniform(-1, 1, (units, units)) * (rng.rand(units, units) < 2 / (units - 1))
        self.W /= np.max(np.abs(np.linalg.eigvals(self.W)))
        self.Win = rng.uniform(-1, 1, (self.driver_width, units)) / np.sqrt(self.driver_width)
        self.bias = rng.uniform(-1, 1, units)

    def feature_sequence(self, values):
        drivers = np.concatenate([values[np.maximum(np.arange(len(values)) - lag, 0)]
                                  for lag in range(self.history + 1)], axis=1)
        state = np.zeros(self.config['n_units'])
        features = np.empty((len(values), self.driver_width + self.config['n_units'] + 1))
        for index, driver in enumerate(drivers):
            proposed = np.tanh(self.config['input_scaling'] * (driver @ self.Win + self.bias)
                               + self.config['spectral_radius'] * (state @ self.W))
            state = (1 - self.config['leak']) * state + self.config['leak'] * proposed
            features[index] = np.r_[driver, state, 1.0]
        return features

    def fit_records(self, records):
        width = self.driver_width + self.config['n_units'] + 1
        gram = self.config['ridge'] * np.eye(width)
        rhs = np.zeros((width, NODES))
        first = SETTINGS['first_anchor']
        for values in records.values():
            features = self.feature_sequence(values)[first:-1]
            target = values[first + 1:]
            gram += features.T @ features
            rhs += features.T @ target
        self.Wout = np.linalg.solve(gram, rhs)
        return self


class HistoryAdapter:
    def __init__(self, model):
        self.model = model
        self.reservoir_units = model.config['n_units']
        self.buffer_width = model.driver_width
        self.n_units = self.reservoir_units + 2 * self.buffer_width
        self.Wout = np.vstack((model.Wout[model.driver_width:-1],
                               np.zeros((self.buffer_width, NODES)),
                               model.Wout[:model.driver_width], model.Wout[-1:]))

    def initial_states(self, values, indices):
        selected, inverse = np.unique(indices, return_inverse=True)
        initials = np.empty((len(selected), self.n_units + 1))
        state = np.zeros(self.reservoir_units)
        driver = np.empty(self.model.driver_width)
        config = self.model.config
        position = 0
        for index in range(selected[-1] + 1):
            driver[:NODES] = values[index]
            for lag in range(1, self.model.history + 1):
                driver[lag * NODES:(lag + 1) * NODES] = values[max(index - lag, 0)]
            proposed = np.tanh(config['input_scaling'] * (driver @ self.model.Win + self.model.bias)
                               + config['spectral_radius'] * (state @ self.model.W))
            state = (1 - config['leak']) * state + config['leak'] * proposed
            if index == selected[position]:
                buffer = np.concatenate([values[max(index - lag, 0)]
                                         for lag in range(self.model.history + 1)])
                initials[position] = np.r_[state, buffer, driver, 1.0]
                position += 1
                if position == len(selected):
                    break
        return initials[inverse]

    def step_batch(self, states, observations, causes=None):
        config = self.model.config
        previous = states[:, self.reservoir_units:self.reservoir_units + self.buffer_width]
        driver = np.concatenate((observations, previous[:, :-NODES]), axis=1)
        if causes is not None:
            rows = np.arange(len(states))
            for lag in range(1, self.model.history + 1):
                driver[rows, lag * NODES + causes] = 0.0
        reservoir = states[:, :self.reservoir_units]
        proposed = np.tanh(config['input_scaling'] * (driver @ self.model.Win + self.model.bias)
                           + config['spectral_radius'] * (reservoir @ self.model.W))
        reservoir = (1 - config['leak']) * reservoir + config['leak'] * proposed
        buffer = np.concatenate((observations, previous[:, :-NODES]), axis=1)
        return np.concatenate((reservoir, buffer, driver, np.ones((len(states), 1))), axis=1)

    def evolve(self, initial, horizon):
        state = initial.copy()
        control = [state @ self.Wout]
        for step in range(horizon):
            state = self.step_batch(state[None, :self.n_units], control[-1][None])[0]
            control.append(state @ self.Wout)
        return np.asarray(control)


def intervention_energies(adapter, initial, control, causes, effects, horizon):
    rows = np.arange(len(causes))
    natural = np.repeat(initial[None, :adapter.n_units], len(causes), axis=0)
    masked = natural.copy()
    predictions = np.repeat(control[0][None], len(causes), axis=0)
    energy = np.zeros(len(causes))
    for step in range(1, horizon + 1):
        natural_next = adapter.step_batch(natural, predictions)
        inputs = predictions.copy()
        inputs[rows, causes] = 0.0
        masked_next = adapter.step_batch(masked, inputs, causes)
        predictions = natural_next @ adapter.Wout
        predictions[rows, effects] = np.sum(masked_next * adapter.Wout[:, effects].T, axis=1)
        energy += (control[step, effects] - predictions[rows, effects]) ** 2
        natural = natural_next[:, :adapter.n_units]
        masked = masked_next[:, :adapter.n_units]
    return energy


def record_anchors(record, length):
    seed = int(np.random.SeedSequence([SETTINGS['history_seed'], record]).generate_state(1)[0])
    return np.random.RandomState(seed).randint(
        SETTINGS['first_anchor'], length - SETTINGS['n_evolution_steps'] - 1,
        size=SETTINGS['n_initial_histories'])


def score_model(model, values, anchors):
    adapter = HistoryAdapter(model)
    initials = adapter.initial_states(values, anchors - 1)
    effects, causes = np.where(EDGE_MASK)
    scores = np.full((len(anchors), NODES, NODES), -1.0)
    scores[:, np.arange(NODES), np.arange(NODES)] = 0.0
    means = []
    horizon = SETTINGS['n_evolution_steps']
    for index, initial in enumerate(initials):
        control = adapter.evolve(initial, horizon)
        source_mean = control.mean(axis=0)
        means.append(source_mean)
        eligible = np.flatnonzero(np.abs(source_mean[causes]) >= SETTINGS['mean_threshold'])
        for start in range(0, len(eligible), SETTINGS['batch_size']):
            selected = eligible[start:start + SETTINGS['batch_size']]
            energy = intervention_energies(adapter, initial, control, causes[selected], effects[selected], horizon)
            scores[index, effects[selected], causes[selected]] = energy / horizon / source_mean[causes[selected]]
    positive = scores > 0
    counts = positive.sum(axis=0)
    ddi = np.divide(np.where(positive, scores, 0).sum(axis=0), counts,
                    out=np.zeros((NODES, NODES)), where=counts > 0)
    return {'ddi': ddi, 'history_scores': scores, 'source_means': np.asarray(means),
            'positive_counts': counts, 'anchors': anchors}


def training_profile(history):
    return {'history_steps': history, 'input': 'state_history', 'output': 'next_state',
            'config': MODEL_CONFIG, 'variables': VARIABLES,
            'train_ids': TRAIN_IDS, 'training_hashes': TRAIN_HASHES,
            'first_anchor': SETTINGS['first_anchor'], 'reservoir_seed': SETTINGS['reservoir_seed'],
            'numpy': np.__version__,
            'implementation': implementation_id(HistoryESN.__init__, HistoryESN.feature_sequence,
                                                HistoryESN.fit_records, model_seed)}


def weight_directory(history):
    profile = training_profile(history)
    folder = RESULT_ROOT / f'history_{history}' / 'weights' / token(profile)[:20]
    save_json(folder / 'manifest.json', profile)
    return folder


def load_or_train(history, index):
    path = weight_directory(history) / f'model_{index + 1:02d}.npz'
    model = HistoryESN.__new__(HistoryESN)
    model.config = dict(MODEL_CONFIG)
    model.history = history
    model.driver_width = NODES * (history + 1)
    if path.exists():
        with np.load(path, allow_pickle=False) as saved:
            for name in ('W', 'Win', 'bias', 'Wout'):
                setattr(model, name, saved[name].copy())
    else:
        model = HistoryESN(MODEL_CONFIG, model_seed(index), history).fit_records(TRAIN)
        save_npz(path, **{name: getattr(model, name) for name in ('W', 'Win', 'bias', 'Wout')})
    return model


def scoring_profile(history):
    return {'training_signature': token(training_profile(history)),
            'settings': {key: SETTINGS[key] for key in ('n_initial_histories', 'n_evolution_steps',
                           'first_anchor', 'mean_threshold', 'history_seed')},
            'formula': 'signed control-source mean; positive-history averaging',
            'implementation': implementation_id(HistoryAdapter.__init__, HistoryAdapter.initial_states,
                HistoryAdapter.step_batch, HistoryAdapter.evolve, intervention_energies, score_model, record_anchors)}


def score_path(history, record, index, values):
    profile = scoring_profile(history)
    folder = RESULT_ROOT / f'history_{history}' / 'scores' / token(profile)[:20]
    save_json(folder / 'manifest.json', profile)
    data_signature = hashlib.sha256(values.tobytes()).hexdigest()[:20]
    return folder / f'record_{record:02d}_{data_signature}' / f'model_{index + 1:02d}.npz'


def run_partition(history, records):
    members = {record: [] for record in records}
    for index in range(SETTINGS['n_models']):
        model = None
        for record, values in records.items():
            path = score_path(history, record, index, values)
            if not path.exists():
                if model is None:
                    model = load_or_train(history, index)
                arrays = score_model(model, values, record_anchors(record, len(values)))
                save_npz(path, **arrays)
            with np.load(path, allow_pickle=False) as saved:
                members[record].append(saved['ddi'].copy())
    return {record: np.stack(matrices) for record, matrices in members.items()}


def pooled_scores(results):
    return np.concatenate([matrices.mean(axis=0)[EDGE_MASK] for matrices in results.values()])


def classification_counts(scores, labels, threshold):
    predicted = scores > threshold
    actual = labels == 1
    tp = int((predicted & actual).sum())
    fp = int((predicted & ~actual).sum())
    fn = int((~predicted & actual).sum())
    return {'TP': tp, 'FP': fp, 'FN': fn,
            'Precision': tp / (tp + fp) if tp + fp else 0.0,
            'Recall': tp / (tp + fn) if tp + fn else 0.0,
            'F1': 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0}


def select_threshold(results):
    scores = pooled_scores(results)
    labels = np.tile(TRUTH[EDGE_MASK], len(results))
    candidates = pd.DataFrame([{'threshold': float(threshold), **classification_counts(scores, labels, threshold)}
                               for threshold in np.unique(np.r_[0.0, scores])])
    best = candidates.sort_values(['F1', 'threshold'], ascending=[False, False]).iloc[0]
    return float(best['threshold']), candidates


REPORT_PROFILE = {'input_blocks': SETTINGS['input_blocks'], 'n_models': SETTINGS['n_models'],
                  'scoring': {str(window): scoring_profile(window - 1) for window in SETTINGS['input_blocks']},
                  'validation_hashes': {str(record): hashlib.sha256(values.tobytes()).hexdigest()
                                        for record, values in VALIDATION.items()},
                  'test_hashes': {str(record): hashlib.sha256(values.tobytes()).hexdigest()
                                  for record, values in TEST.items()},
                  'truth_hash': hashlib.sha256(TRUTH.tobytes()).hexdigest(),
                  'threshold_rule': 'maximum pooled validation F1; ties choose higher absolute threshold',
                  'evaluation_implementation': implementation_id(pooled_scores, classification_counts, select_threshold)}
REPORT_ROOT = RESULT_ROOT / 'tables' / token(REPORT_PROFILE)[:20]
save_json(REPORT_ROOT / 'manifest.json', REPORT_PROFILE)
VALIDATION_RESULTS, THRESHOLDS = {}, {}
with threadpool_limits(limits=SETTINGS['blas_threads']):
    for window in SETTINGS['input_blocks']:
        VALIDATION_RESULTS[window] = run_partition(window - 1, VALIDATION)
        threshold, candidates = select_threshold(VALIDATION_RESULTS[window])
        THRESHOLDS[window] = threshold
        candidates.to_csv(REPORT_ROOT / f'validation_thresholds_W_{window}.csv', index=False)
save_json(REPORT_ROOT / 'thresholds.json', {str(window): threshold for window, threshold in THRESHOLDS.items()})


## 3. Evaluation

Select each window's threshold by maximum pooled validation F1, choosing the higher threshold in a tie. Apply it unchanged to all test records. Pool TP and FP across all non-self directed pairs in the three test records to compute Precision and Recall.


In [ ]:
ROWS, RECORD_ROWS, EDGE_ROWS = [], [], []
with threadpool_limits(limits=SETTINGS['blas_threads']):
    for window in SETTINGS['input_blocks']:
        results = run_partition(window - 1, TEST)
        threshold = THRESHOLDS[window]
        scores = pooled_scores(results)
        labels = np.tile(TRUTH[EDGE_MASK], len(results))
        counts = classification_counts(scores, labels, threshold)
        ROWS.append({'Input blocks W': window, **{key: counts[key] for key in ('TP', 'FP', 'Precision', 'Recall')}})
        for record, matrices in results.items():
            ddi = matrices.mean(axis=0)
            RECORD_ROWS.append({'Input blocks W': window, 'record': record, 'threshold': threshold,
                                **classification_counts(ddi[EDGE_MASK], TRUTH[EDGE_MASK], threshold)})
            EDGE_ROWS.extend({'Input blocks W': window, 'record': record, 'cause': VARIABLES[cause],
                              'effect': VARIABLES[effect], 'DDI': float(ddi[effect, cause]),
                              'true_edge': int(TRUTH[effect, cause]),
                              'predicted_edge': bool(ddi[effect, cause] > threshold)}
                             for effect, cause in zip(*np.where(EDGE_MASK)))
        save_npz(REPORT_ROOT / f'DDI_W_{window}.npz', truth=TRUTH,
                 validation_ids=np.asarray(VALIDATION_IDS), test_ids=np.asarray(TEST_IDS),
                 validation_member_DDI=np.stack(list(VALIDATION_RESULTS[window].values())),
                 test_member_DDI=np.stack(list(results.values())))
SUMMARY = pd.DataFrame(ROWS, columns=['Input blocks W', 'TP', 'FP', 'Precision', 'Recall'])
SUMMARY.to_csv(REPORT_ROOT / 'summary.csv', index=False)
pd.DataFrame(RECORD_ROWS).to_csv(REPORT_ROOT / 'by_record.csv', index=False)
pd.DataFrame(EDGE_ROWS).to_csv(REPORT_ROOT / 'edges.csv', index=False)
TABLE_HTML = SUMMARY.to_html(index=False, formatters={
    'Precision': lambda value: f'{value:.2%}', 'Recall': lambda value: f'{value:.2%}'})
(REPORT_ROOT / 'results.html').write_text('<meta charset="utf-8">' + TABLE_HTML, encoding='utf-8')
display(HTML(TABLE_HTML))
